## Cylindrical-symmetric (2D axisymmetric) band-limited Hellwarth–Nouchi-like toroidal pulse in 0.5–10 GHz.


### This version uses Meep's cylindrical symmetry mode (mp.CYLINDRICAL),

so the toroidal pulse can be simulated much faster in 2D while still capturing
its 3D doughnut topology.


### Now includes 2D snapshot animation of Ez(r,z) during propagation.


Requirements:
- meep (Python bindings with cylindrical symmetry support)
- numpy
- matplotlib
- imageio (for saving animation)


Notes:
- Mapping: 10 GHz -> 1.0 in Meep units (so 0.5 GHz -> 0.05).
- We simulate in cylindrical coordinates (r,z). The pulse symmetry around the axis is enforced.
- Only TM-like variant is coded (E has rho and z components; H has phi component).

In [1]:
import meep as mp
import numpy as np
import matplotlib.pyplot as plt
import os
import imageio


# -------------------------
# Parameters
# -------------------------
f_min_real = 0.5 # GHz
f_max_real = 10.0 # GHz
f_ref_real = 10.0 # GHz -> 1.0 in Meep units


# Normalized frequencies
f_min = f_min_real / f_ref_real # 0.05
f_max = f_max_real / f_ref_real # 1.0
f_center = 0.5 * (f_min + f_max)
f_width = (f_max - f_min) * 0.55


print(f"Normalized center frequency: {f_center:.3f}, bandwidth fwidth={f_width:.3f}")


# Spatial scale parameter of HN pulse
param_a = 0.6


resolution = 30
sr = 12.0 # radial extent
sz = 20.0 # z extent
pml_thickness = 2.0
run_time = 200


# Output folder
outdir = "meep_hn_cyl_output"
os.makedirs(outdir, exist_ok=True)

Normalized center frequency: 0.525, bandwidth fwidth=0.522


### Hellwarth–Nouchi TM envelope in cylindrical coords

In [3]:
def hn_tm_cyl(rho, z, a=param_a):
    denom = (rho**2 + z**2 + a**2)**2.5
    if denom == 0:
        return (0.0, 0.0, 0.0)
    E_rho = 3.0 * rho * z / denom
    E_z = (2.0*z**2 - rho**2 + a**2) / denom
    H_phi = 3.0 * rho * z / denom # c=1 in Meep units
    return (E_rho, E_z, H_phi)


# Source spatial weighting functions (amp_func signature: r (mp.Vector3))
def e_rho_weight(r):
    rho = r.x # in cylindrical mode, x is radial coordinate
    z = r.z
    E_rho, E_z, H_phi = hn_tm_cyl(rho, z)
    return E_rho


def e_z_weight(r):
    rho = r.x
    z = r.z
    E_rho, E_z, H_phi = hn_tm_cyl(rho, z)
    return E_z

### Build cylindrical simulation

In [5]:
# -------------------------
# Build cylindrical simulation
# -------------------------
cell = mp.Vector3(sr, 0, sz) # cylindrical: (r, phi=ignored, z)


sim = mp.Simulation(
cell_size=cell,
resolution=resolution,
dimensions=mp.CYLINDRICAL,
m=0, # azimuthal index
boundary_layers=[mp.PML(pml_thickness)],
)


# Gaussian temporal profile
src_time = mp.GaussianSource(frequency=f_center, fwidth=f_width)


# Sources: drive E_rho and E_z components using HN spatial weighting
sources = [
mp.Source(src=src_time, component=mp.Er, center=mp.Vector3(0, 0, -4), size=mp.Vector3(sr, 0, 6), amp_func=e_rho_weight),
mp.Source(src=src_time, component=mp.Ez, center=mp.Vector3(0, 0, -4), size=mp.Vector3(sr, 0, 6), amp_func=e_z_weight),
]
sim.sources = sources


# Flux region (circle at z=+6)
flux_z = 6.0
flux_reg = mp.FluxRegion(center=mp.Vector3(0, 0, flux_z), size=mp.Vector3(sr-2, 0, 0))
flux = sim.add_flux(f_center, f_width, 100, flux_reg)


# Probe at axis on z=+6
probe_pt = mp.Vector3(0, 0, flux_z)
probe_time, probe_field = [], []


def probe_cb(sim):
    probe_time.append(sim.meep_time)
    probe_field.append(sim.get_field_point(mp.Ez, probe_pt))


# Animation frames
frames = []
def snapshot_cb(sim):
    field = sim.get_array(center=mp.Vector3(), size=cell, component=mp.Ez)
    if field is not None:
        frames.append(field.copy())

### Run simulation

In [6]:
# -------------------------
# Run simulation
# -------------------------
print("Running cylindrical HN pulse sim with snapshots...")
sim.run(mp.at_every(0.5, probe_cb), mp.at_every(5, snapshot_cb), until=run_time)


# Retrieve flux spectrum
freqs = mp.get_flux_freqs(flux)
flux_vals = mp.get_fluxes(flux)

Running cylindrical HN pulse sim with snapshots...
-----------
Initializing structure...
time for choose_chunkdivision = 0.000858068 s
Working in Cylindrical dimensions.
Computational cell is 12 x 0 x 20 with resolution 30
time for set_epsilon = 0.116124 s
-----------


FloatProgress(value=0.0, description='0% done ', max=200.0)

Meep progress: 56.03333333333333/200.0 = 28.0% done in 4.0s, 10.3s to go
on time step 3365 (time=56.0833), 0.00118877 s/step
Meep progress: 112.43333333333334/200.0 = 56.2% done in 8.0s, 6.2s to go
on time step 6749 (time=112.483), 0.00118205 s/step
Meep progress: 170.43333333333334/200.0 = 85.2% done in 12.0s, 2.1s to go
on time step 10229 (time=170.483), 0.00114959 s/step
run 0 finished at t = 200.0 (12000 timesteps)


### save results

In [7]:
# -------------------------
# Save and plot
# -------------------------
np.save(os.path.join(outdir, "probe_time.npy"), np.array(probe_time))
np.save(os.path.join(outdir, "probe_ez.npy"), np.array(probe_field))


# Plot probe time and FFT
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(probe_time, probe_field)
plt.xlabel('time (Meep units)')
plt.ylabel('Ez at probe (axis)')
plt.title('Probe Ez time series')


if len(probe_field) > 4:
    T = probe_time[1] - probe_time[0]
    N = len(probe_field)
    xf = np.fft.rfftfreq(N, d=T)
    yf = np.abs(np.fft.rfft(probe_field))
    plt.subplot(1,2,2)
    plt.plot(xf, yf)
    plt.xlabel('frequency (Meep units)')
    plt.title('Probe spectrum (FFT)')

plt.tight_layout()
plt.savefig(os.path.join(outdir, "probe_time_fft.png"), dpi=150)


# Plot flux spectrum
plt.figure()
plt.plot(freqs, flux_vals)
plt.xlabel('frequency (Meep units)')
plt.ylabel('flux')
plt.title('Flux spectrum at z={}'.format(flux_z))
plt.savefig(os.path.join(outdir, "flux_spectrum.png"), dpi=150)


# Create snapshot animation
if frames:
    print("Saving Ez(r,z) snapshot animation...")
    images = []
    for f in frames:
        fig, ax = plt.subplots()
        im = ax.imshow(f.T, origin='lower', aspect='auto', extent=[-sr/2, sr/2, -sz/2, sz/2])
        ax.set_xlabel('r (Meep units)')
        ax.set_ylabel('z (Meep units)')
        ax.set_title('Ez snapshot')
        fig.colorbar(im, ax=ax)
        plt.close(fig)
        # Render to RGB array
        fig.canvas.draw()
        image = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
        image = image.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        images.append(image)
    imageio.mimsave(os.path.join(outdir, "ez_snapshots.gif"), images, fps=5)
    print("Animation saved to ez_snapshots.gif")


print(f"Outputs saved in {outdir}")

TypeError: cannot pickle 'SwigPyObject' object